# GeneWeaver: Kaggle GPU runner

**Settings → Accelerator: GPU T4 x2, Internet: On.**

Run top to bottom. After every push from the Mac, re-run from the *Get the code* cell (it does `git pull`).

## 1. Check the GPUs
Should list 2 × Tesla T4.

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
import os
os.chdir("/kaggle/working")
if not os.path.exists("GeneWeaver"):
    !git clone https://github.com/shravani-oss/GeneWeaver.git
%cd /kaggle/working/GeneWeaver
!git pull
!git log --oneline -3

## 3. Install

In [ ]:
!pip install -q -r requirements.txt -r requirements-gpu.txt
!pip install -q -e .
!python --version
!python -c "from numba import cuda; print('CUDA available:', cuda.is_available()); cuda.detect()"

## 4. Download chr21 (~46 M bases)

In [ ]:
!mkdir -p data
![ -f data/chr21.fa ] || (wget -q -O data/chr21.fa.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr21.fa.gz && gunzip data/chr21.fa.gz)
!ls -lh data/

## 5. Run all tests on the real GPU
`test_gpu_matches_cpu.py` now runs for real (no simulator), including the chr21 GPU == CPU test.
`-rs` lists anything skipped. Nothing in `test_gpu_matches_cpu.py` should be skipped here.

In [ ]:
!python -m pytest tests -q -rs

## 6. Week 2 check: GPU vs CPU on chr21 through the CLI
Both TSV files must be byte-for-byte identical.

In [ ]:
G = "GAGTCCGAGCAGAAGAAGAA"
for mm in (4, 6):
    !python -m geneweaver scan --fasta data/chr21.fa --guide {G} --max-mm {mm} --backend gpu --chunk-size 5000000 --out results/chr21_gpu_mm{mm}.tsv --quiet
    !python -m geneweaver scan --fasta data/chr21.fa --guide {G} --max-mm {mm} --backend numba-cpu --chunk-size 5000000 --out results/chr21_cpu_mm{mm}.tsv --quiet
    !cmp results/chr21_gpu_mm{mm}.tsv results/chr21_cpu_mm{mm}.tsv && echo "max_mm={mm}: GPU == CPU ✅"

In [ ]:
!head -5 results/chr21_gpu_mm6.tsv | column -t

---
# Mid-project review
**Only run these after sections 5 and 6 pass** (a speedup only counts if GPU results equal CPU results).

## 7. Benchmark: pure Python vs Numba CPU vs GPU
Same machine, same input: a 5 M-base slice of chr21 (20–25 Mb; the first 5 Mb of chr21 is all `N`).
Then the whole chr21 through the real pipeline. Takes ~1–2 minutes. Writes `benchmarks/results.md`.

In [ ]:
!python benchmarks/bench.py data/chr21.fa --full data/chr21.fa --out benchmarks/results.md

In [ ]:
from IPython.display import Markdown
Markdown(open("benchmarks/results.md").read())

## 8. Full genome (hg38) memory test
Downloads hg38 (~940 MB compressed; kept compressed, the parser reads `.gz` directly),
scans chr1–22, X, Y on one GPU with 50 M-base chunks, and logs GPU memory after every chunk.
Reading 3.1 G bases from a `.gz` file takes a while (probably 5–15 minutes).

In [ ]:
![ -f data/hg38.fa.gz ] || wget -q -O data/hg38.fa.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
!ls -lh data/

In [ ]:
!python -m geneweaver scan --fasta data/hg38.fa.gz --guide GAGTCCGAGCAGAAGAAGAA --main-chroms-only \
    --backend gpu --chunk-size 50000000 --out results/hg38_gpu.tsv --chunk-log results/hg38_chunks.tsv --quiet

Pass = no `CUDA_ERROR_OUT_OF_MEMORY` above, and the verdict below says **FLAT**.

In [ ]:
!python benchmarks/memory_report.py results/hg38_chunks.tsv
!head -5 results/hg38_gpu.tsv | column -t